In [1]:
!pip install transformers==4.52.4
!pip install sentence-transformers PyPDF2 faiss-cpu accelerate

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
def generate_text(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [4]:
import re

def is_arabic(text: str) -> bool:
  return bool(re.search(r'[\u0600-\u06FF]', text))

## 2. Extract and Prepare Source Document


In [5]:
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss

In [6]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += (page.extract_text() or "") + "\n"
    for s in ["•", "®", "™"]:
        full_text = full_text.replace(s, "")
    return full_text

## 3. Chunking Strategy


In [7]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

## 4. Embed Chunks and Build a Vector Index


In [8]:
def embed_chunks(chunks, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)
    return model, embeddings

In [9]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

In [10]:
def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [11]:
pdf_path = "/kaggle/input/datasets/mohamedkhaled89/10q-q3-2026-as-filed-pdf/10Q-Q3-2026-as-filed.pdf"

text = extract_text_from_pdf(pdf_path)
chunks = chunk_text(text)
print(f"Document split into {len(chunks)} chunks")

model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

Document split into 116 chunks


In [12]:
def rag_answer(question, k=8):
  is_arg_arabic = is_arabic(question)

  if is_arg_arabic:
    translation_prompt = (
        "Translate the following Arabic financial question into a concise English query. "
        "Use exact 10-Q reporting line items (e.g., 'Total net sales', 'Diluted earnings per share'). "
        "Output ONLY the translated query.\n\n"
        f"Question: {question}"
    )
    query_for_retrieval = generate_text(translation_prompt, max_new_tokens=40).strip()
  else:
    query_for_retrieval = question

  top_chunks = search_index(query_for_retrieval, model_embeddings, index, chunks, k=k)
  context = "\n\n".join(top_chunks)

  if is_arg_arabic:
    language_instruction = "CRITICAL: Write the entire response in ARABIC (باللغة العربية فقط). Keep numbers exactly as written in the text."
  else:
    language_instruction = "Write the entire response in English."

  prompt = (
      "Answer the question using ONLY the provided text.\n"
      f"{language_instruction}\n"
      "- Copy numbers EXACTLY. Never add or change digits.\n"
      "- Do NOT sum numbers across categories. Use the exact line-item value.\n"
      "- Include the unit (millions of dollars).\n"
      "- In tables, the FIRST number is 'Three Months Ended'. Use the first number for quarter questions.\n"
      "- Answer in one or two sentences.\n"
      "- If the exact answer is not stated in the document, respond with 'Not stated in the document' "
      "(Arabic: غير مذكور في المستند).\n\n"
      f"Text:\n{context}\n\n"
      f"Question: {question}"
  )

  return generate_text(prompt, max_new_tokens=250).strip()

In [13]:
questions_en = [
    "What was Total net sales for the Three Months Ended June 27, 2026?",
    "What was iPhone net sales for the Three Months Ended June 27, 2026?",
    "What was Diluted earnings per share for the Three Months Ended June 27, 2026?",
]
questions_ar = [
    'ما هو إجمالي صافي المبيعات (Total net sales) للأشهر الثلاثة المنتهية في 27 يونيو 2026؟',
    'كم كانت مبيعات آيفون (iPhone) للأشهر الثلاثة المنتهية في 27 يونيو 2026؟',
    'ما هي ربحية السهم المخففة (Diluted earnings per share) للأشهر الثلاثة المنتهية في 27 يونيو 2026؟',
]

for q in questions_en + questions_ar:
  print('Q:', q)
  print('A:', rag_answer(q))
  print('-' * 60)

Q: What was Total net sales for the Three Months Ended June 27, 2026?
A: The Total net sales for the Three Months Ended June 27, 2026 was $109,417 million.
------------------------------------------------------------
Q: What was iPhone net sales for the Three Months Ended June 27, 2026?
A: The iPhone net sales for the Three Months Ended June 27, 2026 was $54,252 million.
------------------------------------------------------------
Q: What was Diluted earnings per share for the Three Months Ended June 27, 2026?
A: The diluted earnings per share for the Three Months Ended June 27, 2026 was $2.02.
------------------------------------------------------------
Q: ما هو إجمالي صافي المبيعات (Total net sales) للأشهر الثلاثة المنتهية في 27 يونيو 2026؟
A: إجمالي صافي المبيعات (Total net sales) للأشهر الثلاثة المنتهية في 27 يونيو 2026 هو 109,417 مليون دولار.
------------------------------------------------------------
Q: كم كانت مبيعات آيفون (iPhone) للأشهر الثلاثة المنتهية في 27 يونيو 2026؟
A: ك

In [14]:
import pandas as pd

results = []
for q in questions_en + questions_ar:
    results.append({"Question": q, "Answer": rag_answer(q)})

pd.DataFrame(results)

,Question,Answer
0,What was Total net sales for the Three Months ...,The Total net sales for the Three Months Ended...
1,What was iPhone net sales for the Three Months...,The iPhone net sales for the Three Months Ende...
2,What was Diluted earnings per share for the Th...,The diluted earnings per share for the Three M...
3,ما هو إجمالي صافي المبيعات (Total net sales) ل...,إجمالي صافي المبيعات (Total net sales) للأشهر ...
4,كم كانت مبيعات آيفون (iPhone) للأشهر الثلاثة ا...,كم كانت مبيعات آيفون (iPhone) للأشهر الثلاثة ا...
5,ما هي ربحية السهم المخففة (Diluted earnings pe...,ربحية السهم المخففة (Diluted earnings per shar...


In [15]:
!pip install -q gradio

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [19]:
import gradio as gr
import os

DOCS = {}  # one entry per browser session

def _key(request):
    return getattr(request, "session_hash", "default")

def process_pdf(file, request: gr.Request):
    if file is None:
        return "Please upload a PDF first."
    path = getattr(file, "name", file)
    text = extract_text_from_pdf(path)
    if len(text.strip()) < 50:
        return "No text found. This looks like a scanned PDF (images only), which is not supported yet."
    new_chunks = chunk_text(text)
    emb = model_embeddings.encode(
        new_chunks, convert_to_numpy=True, normalize_embeddings=True, batch_size=64
    )
    DOCS[_key(request)] = {
        "chunks": new_chunks,
        "index": create_faiss_index(emb),
    }
    return f"Ready: {os.path.basename(path)} ({len(new_chunks)} chunks). Ask your question below."

def ask_pdf(question, request: gr.Request):
    doc = DOCS.get(_key(request))
    if doc is None:
        return "Upload a PDF and click 'Process PDF' first.", ""
    if not question or not question.strip():
        return "Type a question.", ""

    arabic = is_arabic(question)
    if arabic:
        t_prompt = (
            "Translate the following Arabic question into a concise English search query. "
            "Keep any English terms exactly as written. Output ONLY the query.\n\n"
            f"Question: {question}"
        )
        query = generate_text(t_prompt, max_new_tokens=40).strip()
    else:
        query = question

    k = min(8, len(doc["chunks"]))
    top = search_index(query, model_embeddings, doc["index"], doc["chunks"], k=k)
    context = "\n\n".join(top)

    lang = (
        "CRITICAL: Write the entire response in ARABIC (باللغة العربية فقط). Keep numbers exactly as written in the text."
        if arabic else "Write the entire response in English."
    )
    prompt = (
        "Answer the question using ONLY the provided text.\n"
        f"{lang}\n"
        "- Copy numbers EXACTLY. Never add or change digits.\n"
        "- Include units and the time period when the text gives them.\n"
        "- If a table has several columns, use the column that matches the question.\n"
        "- Answer in one or two sentences.\n"
        "- If the answer is not stated in the text, respond with 'Not stated in the document' "
        "(Arabic: غير مذكور في المستند).\n\n"
        f"Text:\n{context}\n\n"
        f"Question: {question}"
    )
    answer = generate_text(prompt, max_new_tokens=250).strip()
    sources = "\n\n---\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(top[:3]))
    return answer, sources

with gr.Blocks(title="Bilingual PDF Q&A") as demo:
    gr.Markdown("# 📄 Bilingual PDF Q&A (Arabic + English)\nUpload any PDF, then ask in Arabic or English.")
    with gr.Row():
        pdf_file = gr.File(label="Upload PDF", file_types=[".pdf"])
        with gr.Column():
            process_btn = gr.Button("Process PDF", variant="primary")
            status = gr.Textbox(label="Status", interactive=False)
    question = gr.Textbox(label="Ask in Arabic or English", lines=2)
    ask_btn = gr.Button("Ask", variant="primary")
    answer_box = gr.Textbox(label="Answer", lines=5)
    sources_box = gr.Textbox(label="Sources (top matching chunks)", lines=8)

    process_btn.click(process_pdf, pdf_file, status)
    ask_btn.click(ask_pdf, question, [answer_box, sources_box])
    question.submit(ask_pdf, question, [answer_box, sources_box])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://b15cb5b5d5e9d5ad01.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
